In [22]:
# !pip install -q kokoro>=0.9.4 soundfile
# !apt-get -qq -y install espeak-ng > /dev/null 2>&1
from IPython.display import display, Audio
import soundfile as sf
import sounddevice as sd
import torch

In [ ]:
from kokoro import KPipeline
pipeline = KPipeline(lang_code='a')

🇺🇸 American Female (af_*)
| Voice |	Character |
|--------|-------------|
|af_heart | 	Warm, soft, emotional (default)
|af_bella | 	Expressive, dynamic, one of the best-rated
|af_nicole | 	Professional, clear
|af_jessica | 	Friendly, conversational
|af_sarah | 	Neutral, articulate
|af_sky | 	Bright, energetic
|af_nova | 	Slightly dreamy, gentle
|af_kore | 	Soft, calm
|af_river | 	Relaxed, flowing
|af_alloy | 	Crisp, modern
|af_aoede | 	Musical, lyrical

🇺🇸 American Male (am_*)
|Voice | 	Character|
|--------|-------------|
|am_adam | 	Deep narrator|
|am_michael | 	Natural, casual|
|am_eric | 	Clear, balanced|
|am_liam | 	Youthful|
|am_echo | 	Smooth|
|am_onyx | 	Deeper tone|
|am_fenrir | 	Strong, dramatic|
|am_puck | 	Lighter, energetic|

🇬🇧 British Female (bf_*)
bf_alice
bf_emma
bf_isabella
bf_lily

🇬🇧 British Male (bm_*)
bm_daniel
bm_fable
bm_george
bm_lewis

In [16]:
text = '''
Well buddy, Welcome to the club.
'''
generator = pipeline(text, voice='am_onyx',speed=0.7)
for i, (gs, ps, audio) in enumerate(generator):
    print(i, gs, ps)
    display(Audio(data=audio, rate=24000, autoplay=i==0))
    sf.write(f'{i}.wav', audio, 24000)

am_onyx.pt:   0%|          | 0.00/523k [00:00<?, ?B/s]

0 Well buddy, Welcome to the club. wˈɛl bˈʌdi, wˈɛlkəm tə ðə klˈʌb.


In [23]:
CHARACTER_VOICES = {
    "RAMIREZ": "af_bella",   # Deep, authoritative
    "PATEL": "am_eric",       # Calm, analytical
    "CARTER": "am_michael",   # Natural, conversational
    "ELANA": "af_nicole",      # Expressive female
}
NARRATOR_VOICE = "am_adam"

In [26]:
def speak(text: str, voice: str, speed: float = 0.9):
    """Generate and play a piece of speech."""
    generator = pipeline(text, voice=voice, speed=speed)

    for _, _, audio in generator:
        sd.play(audio, samplerate=24000)
        sd.wait()


def read_screenplay(screenplay: str):
    lines = screenplay.splitlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if not line:
            i += 1
            continue

        # Character cue
        if line.upper() in CHARACTER_VOICES:
            character = line.upper()
            dialogue = []

            i += 1
            while i < len(lines) and lines[i].strip():
                dialogue.append(lines[i].strip())
                i += 1

            speak(
                " ".join(dialogue),
                voice=CHARACTER_VOICES[character],
            )

        # Narration
        else:
            narration = []

            while (
                i < len(lines)
                and lines[i].strip()
                and lines[i].strip().upper() not in CHARACTER_VOICES
            ):
                narration.append(lines[i].strip())
                i += 1

            speak(
                " ".join(narration),
                voice=NARRATOR_VOICE,
                speed=0.95,
            )

# ----------------------------------------------

screenplay = """
    RAMIREZ
I don't know about you,
But wait...

The wind whistles through the air.

    PATEL
I can't believe it!

    ELANA
Run!

The lights flicker.
"""

read_screenplay(screenplay)